# 07 — Évaluation et comparaison avec M-Kiriku

Évalue un modèle sur le test set et le compare à **M-Kiriku** (`AIHubSN/M-Kiriku-ASR`), l'objectif étant de **battre son WER (16,34 %)**. Affichage WER/CER global puis par domaine.


## Prérequis

- **Packages** : exécuter la cellule suivante (installe `ndapuspeech` en mode édition + dépendances).
- **Outils externes** : `ffmpeg`, `yt-dlp`, `lmplz`/`build_binary` (via KenLM) selon le pipeline.
- **Python ≥ 3.11**.


In [ ]:
!pip install -e . --quiet
!pip install torch transformers datasets librosa sentencepiece jiwer peft soundfile pyyaml --quiet

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')


In [ ]:
from ndapuspeech import config
config.ensure_dirs()
print(config.get_config()['config'])
print(config.get_config()['training'])


In [ ]:
from ndapuspeech.data import load_jsonl
from ndapuspeech.evaluation import evaluate_and_compare
from ndapuspeech import config

test_samples = load_jsonl(config.PROCESSED_DATA_DIR / 'ndapuspeech_ds' / 'test.jsonl')
print(f'{len(test_samples)} échantillons de test')


In [ ]:
summaries = evaluate_and_compare(
    models=[
        str(config.TRAINING_OUTPUT_DIR or 'models/ndapuspeech-v1'),  # votre modèle
        'AIHubSN/M-Kiriku-ASR',           # baseline à battre
    ],
    samples=test_samples,
    batch_size=8,
)


In [ ]:
import pandas as pd

rows = [
    {'modèle': m, 'WER': s['overall']['wer'], 'CER': s['overall']['cer']}
    for m, s in summaries.items()
]
df = pd.DataFrame(rows)
display(df)  # ou print(df)

if 'AIHubSN/M-Kiriku-ASR' in summaries:
    baseline = summaries['AIHubSN/M-Kiriku-ASR']['overall']['wer']
    our_model_key = str(config.TRAINING_OUTPUT_DIR or 'models/ndapuspeech-v1')
    if our_model_key in summaries:
        ours = summaries[our_model_key]['overall']['wer']
        print(f'\nVerdict : {"BEATS baseline" if ours < baseline else "sous la baseline"}')
        print(f'M-Kiriku {baseline:.2f}%  vs  NdapuSpeech {ours:.2f}%')


## Étapes suivantes

Transcription de nouveaux fichiers : **08_inference.ipynb**.
